In [ ]:
import os
import json
import shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

!python -m pip install -q -U tinker tinker-cookbook openai_harmony

user_secrets = UserSecretsClient()
os.environ["TINKER_API_KEY"] = user_secrets.get_secret("tinker")

BASE_MODEL = "openai/gpt-oss-20b"

STATE_CHECKPOINT_PATH = (
    "tinker://c5d30e88-db03-5663-a80c-97de25a1174f:train:0/weights/checkpoint-00020"
)

SAMPLER_NAME = "checkpoint-dapo-cot"

WORK = Path("/kaggle/working")
RAW_TINKER_DIR = WORK / "tinker_adapter_raw"
PEFT_ADAPTER_DIR = WORK / "peft_olymp_adapter"
TIKTOKEN_CACHE_DIR = WORK / "tiktoken_cache"
MANIFEST_PATH = WORK / "tinker_export_manifest.json"

for p in [RAW_TINKER_DIR, PEFT_ADAPTER_DIR, TIKTOKEN_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
import tinker

service_client = tinker.ServiceClient()

training_client = await service_client.create_training_client_from_state_async(
    path=STATE_CHECKPOINT_PATH
)

sampler_result = await training_client.save_weights_for_sampler_async(
    SAMPLER_NAME
)

if hasattr(sampler_result, "result_async"):
    sampler_result = await sampler_result.result_async()

SAMPLER_PATH = getattr(sampler_result, "path", None)
if SAMPLER_PATH is None and isinstance(sampler_result, dict):
    SAMPLER_PATH = sampler_result.get("path")

In [ ]:
from tinker_cookbook import weights

if PEFT_ADAPTER_DIR.exists():
    shutil.rmtree(PEFT_ADAPTER_DIR)
PEFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

downloaded_dir = weights.download(
    tinker_path=SAMPLER_PATH,
    output_dir=str(RAW_TINKER_DIR),
)

if PEFT_ADAPTER_DIR.exists():
    shutil.rmtree(PEFT_ADAPTER_DIR)

weights.build_lora_adapter(
    base_model=BASE_MODEL,
    adapter_path=downloaded_dir,
    output_path=str(PEFT_ADAPTER_DIR),
)

In [ ]:
import os
import urllib.request
from pathlib import Path
from openai_harmony import HarmonyEncodingName, load_harmony_encoding

TIKTOKEN_CACHE_DIR = Path("/kaggle/working/tiktoken_encodings")
TIKTOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "o200k_base.tiktoken": "https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken",
    "cl100k_base.tiktoken": "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",
}

for filename, url in FILES.items():
    out = TIKTOKEN_CACHE_DIR / filename
    if out.exists() and out.stat().st_size > 0:
        continue
    urllib.request.urlretrieve(url, out)

os.environ["TIKTOKEN_ENCODINGS_BASE"] = str(TIKTOKEN_CACHE_DIR)
os.environ["TIKTOKEN_RS_CACHE_DIR"] = str(TIKTOKEN_CACHE_DIR)

enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

stop_tokens = list(enc.stop_tokens_for_assistant_actions())

In [ ]:
import os
import json
import shutil
from pathlib import Path

PEFT_ADAPTER_DIR = Path("/kaggle/working/peft_olymp_adapter")
TIKTOKEN_DIR = Path("/kaggle/working/tiktoken_encodings")

EXPORT_DIR = Path("/kaggle/working/offline_lora_bundle")

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

(EXPORT_DIR / "peft_olymp_adapter").mkdir(parents=True)
(EXPORT_DIR / "tiktoken_encodings").mkdir(parents=True)

for name in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy2(PEFT_ADAPTER_DIR / name, EXPORT_DIR / "peft_olymp_adapter" / name)

for name in ["o200k_base.tiktoken", "cl100k_base.tiktoken"]:
    shutil.copy2(TIKTOKEN_DIR / name, EXPORT_DIR / "tiktoken_encodings" / name)

manifest = {
    "base_model": "openai/gpt-oss-20b",
    "bundle_contents": {
        "peft_adapter": "peft_olymp_adapter",
        "tiktoken_encodings": "tiktoken_encodings",
    },
    "files": {},
}

for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        manifest["files"][str(p.relative_to(EXPORT_DIR))] = p.stat().st_size

(EXPORT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

In [ ]:
from pathlib import Path
import shutil
import os

WORK = Path("/kaggle/working")
KEEP = {"offline_lora_bundle", ".virtual_documents"}

for p in WORK.iterdir():
    if p.name in KEEP:
        continue
    if p.is_dir():
        shutil.rmtree(p)
    else:
        p.unlink()

In [ ]:
import shutil

folder_to_zip = '/kaggle/working/offline_lora_bundle' 
archive_name = 'dapo-cot20' 
archive_format = 'zip'

shutil.make_archive(archive_name, archive_format, folder_to_zip)